# Medical Question Summarization with Round-Trip Translation

**Course:** Modern Information Retrieval — University of Tehran
**Author:** Saba Naji

This notebook implements an end-to-end pipeline on the **MeQSum** dataset (1,000 consumer health questions paired with expert-written short summaries):

| Part | Step |
|---|---|
| 1–2 | Load the dataset and clean the text |
| 3 | Remove missing values and questions/summaries with irregular lengths |
| 4–5 | **Round-Trip Translation (RTT)** with MarianMT through 5 pivot languages (es, de, it, zh-CN, fr) |
| 6 | **FQD** — distance-based question diversity (1 − cosine similarity of BERT embeddings) |
| 7 | **PRQD** — Precision-Recall Question Diversity (max F1 over a range of α) |
| 9 | Abstractive summarization with **T5-small** on the merged multilingual question |
| 10 | Evaluation with **ROUGE-1 / ROUGE-2** |

> Recommended runtime: Google Colab with a GPU (T4).

## 0. Setup

In [1]:
!pip install -q rouge-score sentencepiece sacremoses openpyxl

In [2]:
import os
import urllib.request

DATA_DIR = "data"
OUTPUT_DIR = "outputs"
DATA_FILE = os.path.join(DATA_DIR, "MeQSum_ACL2019_BenAbacha_Demner-Fushman.xlsx")
DATA_URL = ("https://raw.githubusercontent.com/abachaa/MeQSum/master/"
            "MeQSum_ACL2019_BenAbacha_Demner-Fushman.xlsx")

os.makedirs(DATA_DIR, exist_ok=True)
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Download the MeQSum dataset (CC BY 4.0) if it is not already present
if not os.path.exists(DATA_FILE):
    urllib.request.urlretrieve(DATA_URL, DATA_FILE)
print("Dataset:", DATA_FILE)

Dataset: data/MeQSum_ACL2019_BenAbacha_Demner-Fushman.xlsx


## Parts 1 & 2 — Loading and cleaning the data

`clean_text` normalizes every question (`CHQ`) and reference summary (`Summary`):
removes HTML tags, removes non-alphanumeric characters, collapses repeated whitespace, and lowercases the text.

In [3]:
import pandas as pd
import re

file_path = DATA_FILE
df = pd.read_excel(file_path)

def clean_text(text):
    if isinstance(text, str):
        text = re.sub(r'<.*?>', '', text)  # Remove HTML tags
        text = re.sub(r'[^a-zA-Z0-9\s]', '', text)  # Remove special characters
        text = re.sub(r'\s+', ' ', text).strip()  # Remove extra spaces
        return text.lower()  # Convert to lowercase
    return text

# Apply preprocessing to questions and summaries
df['CHQ'] = df['CHQ'].apply(clean_text)
df['Summary'] = df['Summary'].apply(clean_text)

print(df.head())

                  File                                                CHQ  \
0  1-131188152.xml.txt  subject who and where to get cetirizine d mess...   
1            14348.txt  who makes bromocriptine i am wondering what co...   
2  1-131985747.xml.txt  subject nulytely message hello can you tell me...   
3            15410.txt  williams syndrome i would like to have my daug...   
4               35.txt  clinicaltrialsgov question general information...   

                                             Summary  
0                        who manufactures cetirizine  
1                     who manufactures bromocriptine  
2          who makes nulytely and where can i buy it  
3  where can i get genetic testing for williams s...  
4  where can i get genetic testing for multiple m...  


## Part 3 — Handling missing and irregular data

* Rows with a missing question or summary are dropped.
* Questions must be **10–300** characters and summaries **5–150** characters; everything outside these ranges is removed.
* The index is reset after filtering.

In [4]:
df.dropna(subset=['CHQ', 'Summary'], inplace=True)

# Define length limits for questions and summaries
MIN_Q_LEN, MAX_Q_LEN = 10, 300
MIN_S_LEN, MAX_S_LEN = 5, 150

# Filter out questions and summaries that are too short or too long
df = df[df['CHQ'].str.len().between(MIN_Q_LEN, MAX_Q_LEN)]
df = df[df['Summary'].str.len().between(MIN_S_LEN, MAX_S_LEN)]

df.reset_index(drop=True, inplace=True)

print("Dataset after handling missing and irregular data:")
print(f"{len(df)} question-summary pairs remain")
print(df.head())

Dataset after handling missing and irregular data:
582 question-summary pairs remain
                  File                                                CHQ  \
0  1-131188152.xml.txt  subject who and where to get cetirizine d mess...   
1  1-131985747.xml.txt  subject nulytely message hello can you tell me...   
2            15410.txt  williams syndrome i would like to have my daug...   
3               21.txt  genetic test for ihhs heart condition is there...   
4  1-136003557.xml.txt  subject friedreichs ataxia message i have been...   

                                             Summary  
0                        who manufactures cetirizine  
1          who makes nulytely and where can i buy it  
2  where can i get genetic testing for williams s...  
3  where can i get genetic testing for ihss in texas  
4  where can i get genetic testing for friedreich...  


## Parts 4 & 5 — Round-Trip Translation (RTT)

Each question is translated **English → pivot language → English**. The five pivot languages are Spanish, German,
Italian, Chinese (Simplified) and French, and the back-translated paraphrases are stored in the columns `CHQ_es`,
`CHQ_de`, `CHQ_it`, `CHQ_zh-CN`, `CHQ_fr`.

**Translation engine.** Translation uses the open-source **MarianMT** models from
[Helsinki-NLP (OPUS-MT)](https://huggingface.co/Helsinki-NLP), which run locally on the GPU. The first version of
this project used the free Google Translate web endpoint (`deep-translator`), but Google rate-limits requests coming
from Colab (HTTP 429), so every translation failed silently. Local models have no request limits and give
reproducible paraphrases.

* Translation runs in batches of 32 sentences of similar length, with beam search (4 beams) on a GPU and greedy
  decoding on a CPU so that the notebook also finishes in reasonable time without a GPU.
* `opus-mt-en-zh` supports several Chinese variants, so the target token `>>cmn_Hans<<` selects Simplified Chinese.
* Back-translations are normalized with the same `clean_text` function as the original questions, so that
  differences come from wording and not from punctuation or casing.
* The cell reports how many questions were actually paraphrased for every language.

In [5]:
import torch
from transformers import MarianMTModel, MarianTokenizer
from tqdm.auto import tqdm

# Define pivot languages
languages = ['es', 'de', 'it', 'zh-CN', 'fr']

# (English -> pivot, pivot -> English) translation models
MODEL_NAMES = {
    'es':    ('Helsinki-NLP/opus-mt-en-es', 'Helsinki-NLP/opus-mt-es-en'),
    'de':    ('Helsinki-NLP/opus-mt-en-de', 'Helsinki-NLP/opus-mt-de-en'),
    'it':    ('Helsinki-NLP/opus-mt-en-it', 'Helsinki-NLP/opus-mt-it-en'),
    'zh-CN': ('Helsinki-NLP/opus-mt-en-zh', 'Helsinki-NLP/opus-mt-zh-en'),
    'fr':    ('Helsinki-NLP/opus-mt-en-fr', 'Helsinki-NLP/opus-mt-fr-en'),
}
TARGET_PREFIX = {'zh-CN': '>>cmn_Hans<< '}  # en-zh is multi-target: select Simplified Chinese

device = "cuda" if torch.cuda.is_available() else "cpu"
BATCH_SIZE = 32
NUM_BEAMS = 4 if device == "cuda" else 1  # beam search on GPU, greedy decoding on CPU (much faster)
print("Device:", device, "| beams:", NUM_BEAMS)

def translate_texts(texts, model_name, prefix="", desc=""):
    # Translate a list of sentences with a MarianMT model, in batches.
    # Sentences are sorted by length so each batch needs little padding; the original order is restored at the end.
    mt_tokenizer = MarianTokenizer.from_pretrained(model_name)
    mt_model = MarianMTModel.from_pretrained(model_name).to(device).eval()
    order = sorted(range(len(texts)), key=lambda i: len(texts[i]))
    outputs = [None] * len(texts)
    for start in tqdm(range(0, len(order), BATCH_SIZE), desc=desc):
        idx = order[start:start + BATCH_SIZE]
        batch = [prefix + texts[i] for i in idx]
        inputs = mt_tokenizer(batch, return_tensors="pt", padding=True,
                              truncation=True, max_length=256).to(device)
        with torch.no_grad():
            generated = mt_model.generate(**inputs, num_beams=NUM_BEAMS, max_new_tokens=256)
        for i, text in zip(idx, mt_tokenizer.batch_decode(generated, skip_special_tokens=True)):
            outputs[i] = text
    del mt_model
    if device == "cuda":
        torch.cuda.empty_cache()
    return outputs

texts = df['CHQ'].tolist()

# Apply RTT for each language
for lang in languages:
    to_pivot, to_english = MODEL_NAMES[lang]
    pivot_texts = translate_texts(texts, to_pivot, TARGET_PREFIX.get(lang, ""), desc=f"en -> {lang}")
    final_texts = translate_texts(pivot_texts, to_english, desc=f"{lang} -> en")

    df[f'CHQ_{lang}'] = [clean_text(t) for t in final_texts]
    n_changed = int((df[f'CHQ_{lang}'] != df['CHQ']).sum())
    print(f"{lang:>6}: {n_changed}/{len(df)} questions paraphrased\n")

# Save updated dataset
df.to_csv(os.path.join(OUTPUT_DIR, "translated_dataset.csv"), index=False)
print("Round-trip translation completed")

Device: cpu | beams: 1
    es: 574/582 questions paraphrased

    de: 576/582 questions paraphrased

    it: 581/582 questions paraphrased

 zh-CN: 582/582 questions paraphrased

    fr: 577/582 questions paraphrased

Round-trip translation completed


In [6]:
# Example: one question and its five round-trip paraphrases
example = df.iloc[0]
print("Original:", example['CHQ'], "\n")
for lang in languages:
    print(f"{lang:>6}:", example[f'CHQ_{lang}'])

Original: subject who and where to get cetirizine d message i needwant to know who manufscturs cetirizine my walmart is looking for a new supply and are not getting the recent 

    es: subject who and where to get cetirizine d message i need to know who manufscturs cetirizine my walmart is looking for a new offer and they are not getting the recent
    de: subject who and where cetirizine d to get message i need to know who manufactures cetirizine my walmart is looking for a new offer and not get the latest
    it: object who and where to get message cetirillina i need to know who produces cetirillina my walkart is looking for a new supply and they are not getting the recent
 zh-CN: ask who and where to get the message from cetrizine d i need to know who manufstrs ceterizine is looking for new supplies and not getting the closest
    fr: topic who and where to get thisirizine message i want to know who manufsctors cetirizine my valmart is looking for a new offer and do not receive the

### Safeguard: empty paraphrases

A back-translation could in principle become empty after cleaning (for example, if a model returned only non-Latin
characters, which `clean_text` removes), and BERT cannot embed a missing value. Any empty paraphrase is replaced with
the original question. In this run no replacement was needed.

In [11]:
df = pd.read_csv(os.path.join(OUTPUT_DIR, "translated_dataset.csv"))
for lang in languages:
    col = f'CHQ_{lang}'
    empty = df[col].isna() | (df[col].astype(str).str.strip() == "")
    df.loc[empty, col] = df.loc[empty, 'CHQ']
    print(f"{lang:>6}: {int(empty.sum())} empty paraphrases replaced")
df.to_csv(os.path.join(OUTPUT_DIR, "translated_dataset.csv"), index=False)

    es: 0 empty paraphrases replaced
    de: 0 empty paraphrases replaced
    it: 0 empty paraphrases replaced
 zh-CN: 0 empty paraphrases replaced
    fr: 0 empty paraphrases replaced


## Part 6 — FQD (distance-based question diversity)

Every question $Q$ and paraphrase $\hat{Q}$ is embedded with **BERT** (`bert-base-uncased`, mean-pooled last hidden state):

$$d_{FQD}(Q,\hat{Q}) = 1 - \frac{h_Q \cdot h_{\hat{Q}}}{\lVert h_Q\rVert \, \lVert h_{\hat{Q}}\rVert}$$

The score of a question is the mean distance over its five paraphrases. Scores are min-max normalized to $[0,1]$
and the questions with $\mu_1 < FQD < \mu_2$ (0.2 and 0.8) are kept — paraphrases that are neither near-copies
nor semantically drifted.

In [12]:
import numpy as np
import torch
from transformers import BertModel, BertTokenizer
import pandas as pd

# Load BERT model & tokenizer
bert_model = BertModel.from_pretrained("bert-base-uncased")
tokenizer = BertTokenizer.from_pretrained("bert-base-uncased")

def get_embedding(text):
    tokens = tokenizer(text, return_tensors="pt", padding=True, truncation=True, max_length=128)
    with torch.no_grad():
        output = bert_model(**tokens)
    return output.last_hidden_state.mean(dim=1).squeeze().numpy()

def compute_dFQD(h_Q, h_Q_hat):
    dot_product = np.dot(h_Q, h_Q_hat)
    norm_h_Q = np.linalg.norm(h_Q)
    norm_h_Q_hat = np.linalg.norm(h_Q_hat)

    if norm_h_Q == 0 or norm_h_Q_hat == 0:
        return 1

    cosine_similarity = dot_product / (norm_h_Q * norm_h_Q_hat)
    dFQD_value = 1 - cosine_similarity

    return dFQD_value

def normalize_FQD(dFQD_values):
    min_dFQD = min(dFQD_values)
    max_dFQD = max(dFQD_values)

    if max_dFQD - min_dFQD == 0:
        return [0] * len(dFQD_values)

    normalized_FQD = [(d - min_dFQD) / (max_dFQD - min_dFQD) for d in dFQD_values]
    return normalized_FQD

# Load dataset
df = pd.read_csv(os.path.join(OUTPUT_DIR, "translated_dataset.csv"))

# List of languages for translation columns
languages = ['es', 'de', 'it', 'zh-CN', 'fr']

# Compute embeddings and dFQD for each row, comparing the original question with all translations
dFQD_values = []
for index, row in df.iterrows():
    h_Q = get_embedding(row["CHQ"])
    distances = []

    # Calculate the dFQD between the original question and each translation
    for lang in languages:
        h_Q_hat = get_embedding(row[f"CHQ_{lang}"])
        distances.append(compute_dFQD(h_Q, h_Q_hat))

    # Calculate the average dFQD score for this row
    dFQD_values.append(np.mean(distances))

# Normalize FQD values
df["FQD"] = normalize_FQD(dFQD_values)

# Select a subset of the dataset based on optimal FQD range
mu1, mu2 = 0.2, 0.8
df_filtered = df[(df["FQD"] > mu1) & (df["FQD"] < mu2)]

# Save final dataset
df_filtered.to_csv(os.path.join(OUTPUT_DIR, "filtered_dataset.csv"), index=False)
print(f"Raw mean dFQD: {np.mean(dFQD_values):.4f}")
print(f"Selected {len(df_filtered)}/{len(df)} questions with {mu1} < FQD < {mu2}")
print("FQD calculation completed")

Raw mean dFQD: 0.0749
Selected 302/582 questions with 0.2 < FQD < 0.8
FQD calculation completed


## Part 7 — PRQD (precision-recall question diversity)

For a scaling factor $\alpha$, precision and recall between the embeddings are

$$\text{prec}(\alpha) = \sum_{v} \min\big(\alpha\, h_Q(v),\ h_{\hat{Q}}(v)\big), \qquad
\text{rec}(\alpha) = \sum_{v} \min\Big(h_Q(v),\ \frac{h_{\hat{Q}}(v)}{\alpha}\Big)$$

$\alpha$ is swept over 10 values in $[0.1, 2]$, and PRQD is the **maximum F1** over this curve. The mean over the five
paraphrases is min-max normalized and questions with $0.2 < PRQD < 0.8$ are selected.

In [14]:
import numpy as np
import torch
from transformers import BertModel, BertTokenizer
import pandas as pd

bert_model = BertModel.from_pretrained("bert-base-uncased")
tokenizer = BertTokenizer.from_pretrained("bert-base-uncased")

def get_embedding(text):
    tokens = tokenizer(text, return_tensors="pt", padding=True, truncation=True, max_length=128)
    with torch.no_grad():
        output = bert_model(**tokens)
    return output.last_hidden_state.mean(dim=1).squeeze().numpy()


def compute_precision_recall(hQ, hQ_hat, alpha_values):
    precision_scores = []
    recall_scores = []

    for alpha in alpha_values:
        precision = sum(min(alpha * hQ[v], hQ_hat[v]) for v in range(len(hQ)))
        recall = sum(min(hQ[v], hQ_hat[v] / alpha) for v in range(len(hQ_hat)))

        precision_scores.append(precision)
        recall_scores.append(recall)

    return precision_scores, recall_scores

def compute_PRQD(hQ, hQ_hat):
    alpha_values = np.linspace(0.1, 2, 10)  # Define range of alpha values
    precision_scores, recall_scores = compute_precision_recall(hQ, hQ_hat, alpha_values)

    f1_scores = [
        (2 * prec * rec) / (prec + rec) if (prec + rec) != 0 else 0
        for prec, rec in zip(precision_scores, recall_scores)
    ]

    return max(f1_scores)

df = pd.read_csv(os.path.join(OUTPUT_DIR, "translated_dataset.csv"))

languages = ['es', 'de', 'it', 'zh-CN', 'fr']

# Compute PRQD scores for each row, comparing the original question with all translations
prqd_scores = []
for index, row in df.iterrows():
    hQ = get_embedding(row["CHQ"])
    scores = []

    for lang in languages:
        hQ_hat = get_embedding(row[f"CHQ_{lang}"])
        scores.append(compute_PRQD(hQ, hQ_hat))

    prqd_scores.append(np.mean(scores))

# Normalize PRQD scores
min_prqd, max_prqd = min(prqd_scores), max(prqd_scores)
df["PRQD_Score"] = [(s - min_prqd) / (max_prqd - min_prqd) for s in prqd_scores]

# Select a subset based on PRQD score threshold
mu1, mu2 = 0.2, 0.8
df_filtered = df[(df["PRQD_Score"] > mu1) & (df["PRQD_Score"] < mu2)]

# Save final dataset
df_filtered.to_csv(os.path.join(OUTPUT_DIR, "filtered_dataset2.csv"), index=False)
print(f"Selected {len(df_filtered)}/{len(df)} questions with {mu1} < PRQD < {mu2}")
print("PRQD calculation completed")

Selected 464/582 questions with 0.2 < PRQD < 0.8
PRQD calculation completed


## Part 9 — Summarization with T5-small

The original question and its five round-trip paraphrases are concatenated into `Merged_CHQ`, so the model sees
several phrasings of the same information need. **T5-small** (with the `summarize:` task prefix, input truncated to
256 tokens) generates a summary of at most 30 tokens using beam search (4 beams). The GPU is used when available.

In [15]:
import pandas as pd
import torch
from transformers import T5Tokenizer, T5ForConditionalGeneration

model_name = "t5-small"
tokenizer = T5Tokenizer.from_pretrained(model_name)
device = "cuda" if torch.cuda.is_available() else "cpu"
model = T5ForConditionalGeneration.from_pretrained(model_name).to(device)

df = pd.read_csv(os.path.join(OUTPUT_DIR, "translated_dataset.csv"))

def summarize(text):
    if not isinstance(text, str) or text.strip() == "":
        return ""
    inputs = tokenizer("summarize: " + text, return_tensors="pt", max_length=256, truncation=True).to(device)
    summary_ids = model.generate(inputs.input_ids, max_length=30, num_beams=4, early_stopping=True)
    return tokenizer.decode(summary_ids[0], skip_special_tokens=True)

df["Merged_CHQ"] = df[["CHQ", "CHQ_es", "CHQ_de", "CHQ_it", "CHQ_zh-CN", "CHQ_fr"]].fillna("").agg(" ".join, axis=1).str.strip()
df["Summary2"] = df["Merged_CHQ"].apply(summarize)

df.to_csv(os.path.join(OUTPUT_DIR, "summarized_dataset.csv"), index=False)
print("Summarization completed.")

Summarization completed.


## Part 10 — Evaluation with ROUGE

Generated summaries (`Summary2`) are compared with the expert reference summaries (`Summary`) using the F1 score of
**ROUGE-1** (unigram overlap) and **ROUGE-2** (bigram overlap), with Porter stemming.

In [16]:
import pandas as pd
from rouge_score import rouge_scorer

df = pd.read_csv(os.path.join(OUTPUT_DIR, "summarized_dataset.csv"))

scorer = rouge_scorer.RougeScorer(['rouge1', 'rouge2'], use_stemmer=True)

def evaluate_summary(original, generated):
    scores = scorer.score(original, generated)
    return scores['rouge1'].fmeasure, scores['rouge2'].fmeasure

df[['ROUGE-1', 'ROUGE-2']] = df.apply(lambda row: evaluate_summary(row['Summary'], row['Summary2']), axis=1, result_type="expand")

avg_rouge1 = df['ROUGE-1'].mean()
avg_rouge2 = df['ROUGE-2'].mean()

print(f"Average ROUGE-1 Score: {avg_rouge1:.4f}")
print(f"Average ROUGE-2 Score: {avg_rouge2:.4f}")

df.to_csv(os.path.join(OUTPUT_DIR, "evaluated_summaries.csv"), index=False)

Average ROUGE-1 Score: 0.2441
Average ROUGE-2 Score: 0.0909


In [17]:
# A few generated summaries next to the reference summaries
pd.set_option("display.max_colwidth", 120)
df[["CHQ", "Summary", "Summary2", "ROUGE-1"]].head(5)

,CHQ,Summary,Summary2,ROUGE-1
0,subject who and where to get cetirizine d message i needwant to know who manufscturs cetirizine my walmart is lookin...,who manufactures cetirizine,subject who and where to get cetirizine d message i needwant to know who manufscturs ce,0.210526
1,subject nulytely message hello can you tell me where do i order the nulytely who is the manufacture what phone numbe...,who makes nulytely and where can i buy it,subject nulytely message hello can you tell me where do i order the nulytely who is the manufacture what phone numbe...,0.322581
2,williams syndrome i would like to have my daughter tested for williams syndrome could you please tell me where i wou...,where can i get genetic testing for williams syndrome,williams syndrome i would like my daughter to be tested for williams syndrome could you please tell me where i would,0.400000
3,genetic test for ihhs heart condition is there a commercial genetic test for the ihhs heart condition my family suff...,where can i get genetic testing for ihss in texas,genetic test for ihhs heart condition is there a commercial genetic test for the ihhs heart condition my family suffer,0.200000
4,subject friedreichs ataxia message i have been told i have friedreichs ataxia i am looking for a treatment to revers...,where can i get genetic testing for friedreichs and what are the treatments for it,subject friedreichs ataxia message i have been told i have friedreichs ataxia i am looking for,0.193548
